1. Start with a pretrained LLM

Suppose we have a base model:

$$ \pi_{\text{base}}(y|x) $$

It knows language, facts, patterns, etc., but it isn't necessarily a good assistant.

Give it:

"Explain why the sky is blue."

It may produce a technically correct answer, but it doesn't inherently know whether we prefer:

concise vs verbose
helpful vs evasive
safe vs unsafe
rigorous vs hand-wavy
conversational vs robotic

That's the alignment problem.

2. SFT comes first

We usually first do Supervised Fine-Tuning.

We have examples:

$$ (x,y^*) $$

where humans demonstrate a desirable answer.

Training is simply:

$$ \boxed{ \max_\theta \log\pi_\theta(y^*|x) } $$

So if the human wrote:

"The sky appears blue because..."

we train the model to increase the probability of those tokens.

After SFT:

$$ \pi_{\text{base}} \rightarrow \pi_{\text{SFT}} $$

Now we have a reasonably useful instruction-following model.

3. But SFT has a limitation

Imagine we have:

Prompt:
"Explain quantum mechanics to a 10-year-old."

Model generates:

Answer A

Quantum mechanics describes how matter and energy behave at very small scales. Think of...

Answer B

Quantum mechanics is the study of quantum phenomena.

A human might say:

$$ A \succ B $$

But both are valid training targets.

SFT needs us to say:

"Train on A."

It doesn't naturally encode:

"A is better than B."

That's a fundamentally different kind of information.

4. Preference data gives us that information

Instead of demonstrations:

$$ (x,y^*) $$

we collect comparisons:

$$ \boxed{ (x,y_w,y_l) } $$

where:

\(y_w\) = winner / chosen
\(y_l\) = loser / rejected

Example:

Prompt:
Explain quantum mechanics to a 10-year-old.

Chosen:
"Imagine everything is made of tiny LEGO pieces..."

Rejected:
"Quantum mechanics is a mathematical framework..."

The human isn't necessarily saying:

"This answer deserves reward 8.73."

They're saying:

"I prefer this one."

That's a much weaker but very useful signal.

5. This is what classical RLHF did

The classic InstructGPT pipeline was roughly:

$$ \boxed{ \text{SFT} \rightarrow \text{Reward Model} \rightarrow \text{PPO} } $$

OpenAI's InstructGPT work explicitly used demonstrations for SFT, human comparisons to train a reward model, and then PPO to optimize the policy against that reward model.

Conceptually:

                 Human demonstrations
                         ↓
                      SFT
                         ↓
                    π_SFT
                         ↓
              generate responses
                         ↓
                Human comparisons
                         ↓
                  Reward Model
                         ↓
                    PPO / RL
                         ↓
                  aligned policy
6. What does the Reward Model do?

Suppose humans say:

$$ y_A\succ y_B $$

We train a reward model:

$$ r_\phi(x,y) $$

to produce:

$$ r_\phi(x,y_A)>r_\phi(x,y_B) $$

So instead of asking a human every time, we now have a learned judge.

For example:

Response A → reward = 2.8
Response B → reward = -0.7

PPO then tries to make the policy generate responses with higher reward.

7. And here's the problem DPO attacks

The classical pipeline has a lot of machinery:

Preference data
      ↓
Reward Model
      ↓
Generate samples from policy
      ↓
Reward them
      ↓
PPO
      ↓
Update policy
      ↓
Generate again
      ↓
...

DPO asks:

Do we actually need to separately train a reward model and then run online PPO?

The DPO paper shows that under the KL-regularized RL formulation, you can transform the problem so the policy can be trained directly from preference pairs.

So:

$$ \boxed{ \text{Preference pairs} \rightarrow \text{DPO} \rightarrow \text{new policy} } $$

No separately trained reward model.

No online rollout loop.

That's the big conceptual jump.

8. Don't misunderstand DPO

DPO is not:

"SFT, but with two answers."

It is still fundamentally different.

SFT says:

$$ \boxed{ \text{increase probability of this answer} } $$

DPO says:

$$ \boxed{ \text{make the preferred answer relatively more likely than the rejected answer} } $$

That word relatively becomes extremely important.

And that's exactly why the reference model appears later.

9. Where KTO enters

DPO assumes you have:

$$ (x,y_w,y_l) $$

A preference pair.

But imagine a production system where users simply give feedback:

User 1:
response → 👍

User 2:
response → 👎

User 3:
response → 👍

There's no requirement that we know:

"For this exact prompt, response A was preferred over response B."

We only know:

$$ (x,y,label) $$

where:

$$ label\in\{\text{desirable},\text{undesirable}\} $$

That's the setting KTO targets.

So:

$$ \boxed{ \text{DPO}:\text{pairwise preference} } $$ $$ \boxed{ \text{KTO}:\text{binary desirability feedback} } $$
10. The map of Module 2

Now the reason for our next lessons should be clear:

SFT
 │
 │ "this is a good example"
 ↓
Preference data
 │
 ├───────────────┐
 │               │
paired          unpaired
 │               │
 ↓               ↓
DPO             KTO

And DPO itself comes from:

$$ \text{KL-constrained RLHF} \rightarrow \text{optimal policy} \rightarrow \text{preference objective} $$

That is what we'll derive next.

One important connection to your PPO learning

You already learned that in RLHF PPO we have a reference policy and a KL constraint:

$$ r(x,y)-\beta \log\frac{\pi_\theta(y|x)} {\pi_{\text{ref}}(y|x)} $$

That same idea is not random decoration in DPO.

DPO essentially exploits the mathematical structure of that KL-regularized RL problem to eliminate the explicit reward-model + PPO optimization stage.

So we're not jumping from RL to some unrelated LLM trick.

We're going:

PPO/RLHF→understand the constrained objective→DPO removes the need for explicit RL optimization